# This a demo for executorch usage

## Install the executorch library

In [ ]:
!pip install executorch==0.6.0

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.8/7.8 MB 13.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 865.0/865.0 MB 1.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.5/3.5 MB 89.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.4/7.4 MB 111.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 393.1/393.1 MB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 101.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 69.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 897.7/897.7 kB 46.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 571.0/571.0 MB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 200.2/200.2 MB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 54.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 22.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 158.2/158.

In [ ]:
import torch

torch.__version__

'2.7.0+cu126'

In [ ]:
from executorch import version

version.__version__

'0.6.0+cpu'

In [ ]:
from executorch.runtime import Runtime

runtime = Runtime.get()

operator_names = runtime.operator_registry.operator_names

print(len(operator_names))

217


## Load the example model file

In [ ]:
from torchvision.models import mobilenet_v2
from torchvision.models.mobilenetv2 import MobileNet_V2_Weights

mv2 = mobilenet_v2(weights=MobileNet_V2_Weights.DEFAULT) # This is torch.nn.Module

Downloading: "https://download.pytorch.org/models/mobilenet_v2-7ebf99e0.pth" to /root/.cache/torch/hub/checkpoints/mobilenet_v2-7ebf99e0.pth


100%|██████████| 13.6M/13.6M [00:00<00:00, 59.5MB/s]


## Export the model

In [ ]:
from __future__ import print_function

from executorch.exir import to_edge
from executorch.backends.xnnpack.partition.xnnpack_partitioner import XnnpackPartitioner

model = mv2.eval() # turn into evaluation mode

example_inputs = (torch.randn((1, 3, 224, 224)),) # Necessary for exporting the model

exported_graph = torch.export.export(model, example_inputs) # Core Aten graph

print(exported_graph)

ExportedProgram:
    class GraphModule(torch.nn.Module):
        def forward(self, p_features_0_0_weight: "f32[32, 3, 3, 3]", p_features_0_1_weight: "f32[32]", p_features_0_1_bias: "f32[32]", p_getattr_l__self___features___1___conv_0_0_weight: "f32[32, 1, 3, 3]", p_getattr_l__self___features___1___conv_0_1_weight: "f32[32]", p_getattr_l__self___features___1___conv_0_1_bias: "f32[32]", p_getattr_l__self___features___1___conv_1_weight: "f32[16, 32, 1, 1]", p_getattr_l__self___features___1___conv_2_weight: "f32[16]", p_getattr_l__self___features___1___conv_2_bias: "f32[16]", p_getattr_l__self___features___2___conv_0_0_weight: "f32[96, 16, 1, 1]", p_getattr_l__self___features___2___conv_0_1_weight: "f32[96]", p_getattr_l__self___features___2___conv_0_1_bias: "f32[96]", p_getattr_l__self___features___2___conv_1_0_weight: "f32[96, 1, 3, 3]", p_getattr_l__self___features___2___conv_1_1_weight: "f32[96]", p_getattr_l__self___features___2___conv_1_1_bias: "f32[96]", p_getattr_l__self___features

## Serialise the model

In [ ]:
from executorch.exir import to_edge_transform_and_lower

executorch_program = to_edge_transform_and_lower(
    exported_graph,
    partitioner = [XnnpackPartitioner()] # For accelerated CPU. You can swap other backends
).to_executorch()

In [ ]:
pte_path = "mv2_xnnpack.pte"

with open(pte_path, "wb") as file:
    executorch_program.write_to_file(file) # Serializing into .pte file

## Excute the model

In [ ]:
program = runtime.load_program(pte_path)
method = program.load_method("forward")

[program.cpp:135] InternalConsistency verification requested but not available


In [ ]:
t = torch.randn((1, 3, 224, 224))

output = method.execute([t])
assert len(output) == 1, f"Unexpected output length {len(output)}"
assert output[0].size() == torch.Size([1, 1000]), f"Unexpected output size {output[0].size()}"
print("PASS")

PASS
